# Day 35: Autonomous Research Assistant

An agent that searches the web, reads pages, and summarises findings.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 1. Define tools

In [ ]:
search = TavilySearchResults(max_results=3)

@tool
def scrape_website(url: str) -> str:
    """Fetch and extract main text content from a URL."""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator='\n', strip=True)
        # Limit to first 5000 chars
        return text[:5000]
    except Exception as e:
        return f"Error scraping {url}: {e}"

tools = [search, scrape_website]

## 2. Build the agent

In [ ]:
prompt = PromptTemplate.from_template("""You are a research assistant. Given a question, follow these steps:
1. Use 'tavily_search_results_json' to find relevant articles.
2. For the top results, use 'scrape_website' to get full text.
3. Synthesise a comprehensive answer with citations.

Question: {input}

{agent_scratchpad}
""")

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

## 3. Run research assistant

In [ ]:
query = "What are the latest breakthroughs in solar panel efficiency?"
result = agent_executor.invoke({"input": query})
print("\n=== FINAL REPORT ===\n")
print(result["output"])

## 4. Alternative: manual pipeline (without agent loop)
For more control, implement a sequential process: search → scrape → summarise → combine.

In [ ]:
def research_pipeline(query: str):
    # Step 1: Search
    search_results = search.invoke(query)
    urls = [result['url'] for result in search_results]
    
    # Step 2: Scrape each
    contents = []
    for url in urls:
        text = scrape_website(url)
        contents.append({"url": url, "text": text})
    
    # Step 3: Summarise each (optional)
    summaries = []
    for content in contents:
        summary = llm.invoke(f"Summarise this in 3 sentences:\n{content['text']}").content
        summaries.append(f"Source: {content['url']}\nSummary: {summary}")
    
    # Step 4: Combine into final report
    combined = "\n\n".join(summaries)
    final = llm.invoke(f"Based on these summaries, answer: {query}\n\n{combined}").content
    return final

print(research_pipeline("Latest advances in battery technology"))